<img src="https://datascientest.fr/train/assets/logo_liora.png" style="height:150px">

<hr style="border-width:2px;border-color:#75DFC1">
<center><h1> AI Engineers — Démo Part 11 </h1></center>
<center><h2> Brancher le RAG sur le monde : MCP </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

> **Part 11 · 15 min · ≈ 0,01 $ · bloc facultatif de la Part 11 · Avant la séance : <code>make index-titled</code> (index Qdrant)**
> **Fermez les autres notebooks** : Qdrant local = 1 processus. **Sans gateway ni Qdrant, la démo se rabat sur les sorties enregistrées.**

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 1. Ce qu'est MCP </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **MCP** = un **protocole standard** entre une application d'IA et des **outils**.
- Écrit **une fois**, utilisable par **tout client** : Claude Desktop, Claude Code, un agent LangChain…

```
  CLIENT (l'appli d'IA)                SERVEUR MCP (le nôtre)         OUTILS
 ┌─────────────────────┐   stdio     ┌─────────────────────┐     ┌──────────────┐
 │ Claude Desktop      │ ◀─────────▶ │ src/mcp_server.py   │ ──▶ │ search_fiches│ ─▶ Qdrant
 │ Claude Code         │  JSON-RPC   │                     │     │ ask_fiches   │ ─▶ + modèle
 │ agent LangChain     │             │ "voici mes outils"  │     └──────────────┘
 └─────────────────────┘             └─────────────────────┘
        1. tools/list  →  la liste des outils, avec leur description
        2. tools/call  →  l'appel d'un outil, le résultat revient en texte
```

- **La description de l'outil** est ce que le modèle lit pour choisir : elle compte autant que le code.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 2. Le serveur du projet </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">



In [1]:
import os

# Move to the repository root when Jupyter starts the kernel in the demo folder.
if os.path.basename(os.getcwd()) == "demo":
    os.chdir("../..")

from pathlib import Path

print(Path("src/mcp_server.py").read_text(encoding="utf-8"))

"""MCP server (stdio, official `mcp` SDK, FastMCP) that exposes the RAG to any MCP client: search the fiches, or ask a sourced question."""
from mcp.server.fastmcp import FastMCP

from src.generation import answer_question, build_context, check_sources
from src.ingestion import load_sections
from src.retrieval import expand_to_sections, search

mcp = FastMCP("fiches-conges")
SECTIONS = load_sections()


@mcp.tool()
def search_fiches(query: str, k: int = 3) -> str:
    """Search the fiches on employee leave (Service-Public). Returns the k best sections, each preceded by its fiche id."""
    docs = expand_to_sections(search(query, k=k, collection="chunks_titled"), SECTIONS)
    return build_context(docs)


@mcp.tool()
def ask_fiches(question: str) -> str:
    """Answer a question on employee leave from the fiches, with the ids of the fiches used (or NO_ANSWER)."""
    answer, docs = answer_question(question, sections=SECTIONS)
    invented = check_sources(answer, docs)
    text = answer.

- **Deux outils**, un décorateur `@mcp.tool()` chacun : la fonction Python **devient** l'outil.
- Le nom, les types et la **docstring** deviennent le **schéma** que le client voit.
- `make mcp` le lance : il **attend** un client sur stdin/stdout, il n'affiche rien.
- ⚠️ Jamais de `print` dans un serveur stdio : stdout **est** le protocole.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 3. Un client Python : lister puis appeler </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">



In [2]:
import os
import logging

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

logging.getLogger("httpx").setLevel(logging.WARNING)

server = StdioServerParameters(command="uv", args=["run", "python", "-m", "src.mcp_server"], cwd=str(Path.cwd()))
devnull = open(os.devnull, "w")  # the server logs go to stderr: not on the projector

try:
    async with stdio_client(server, errlog=devnull) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listing = await session.list_tools()
            for tool in listing.tools:
                print("Outil :", tool.name)
                print("   ", tool.description)
                print("    paramètres :", list(tool.inputSchema["properties"]))
            names = {tool.name for tool in listing.tools}
            print("\n✅ 2 outils exposés" if names == {"search_fiches", "ask_fiches"} else "\n❌ outils inattendus : " + str(names))

            result = await session.call_tool("search_fiches", {"query": "congé pour mon mariage", "k": 2})
            text = result.content[0].text
            print("\n--- search_fiches : début du résultat ---")
            print(text[:350], "[...]")
            print("\n✅ la fiche du mariage (F34154) est trouvée" if "[F34154]" in text else "\n❌ F34154 absente")
except Exception as error:
    print("⚠️ serveur, gateway ou Qdrant inaccessible :", type(error).__name__)
    print("Sortie enregistrée : 2 outils (search_fiches : query, k | ask_fiches : question) ; recherche « congé pour mon mariage » → [F34154] Congé du salarié pour mariage ou Pacs en premier.")

Outil : search_fiches
    Search the fiches on employee leave (Service-Public). Returns the k best sections, each preceded by its fiche id.
    paramètres : ['query', 'k']
Outil : ask_fiches
    Answer a question on employee leave from the fiches, with the ids of the fiches used (or NO_ANSWER).
    paramètres : ['question']

✅ 2 outils exposés



--- search_fiches : début du résultat ---
[F34154]
[Congé du salarié pour mariage ou Pacs]
Source : https://www.service-public.gouv.fr/particuliers/vosdroits/F34154 (mise à jour : 2026-03-27)  
Le salarié bénéficie de jours de congé à l’occasion de son mariage ou à la conclusion d’un Pacs. Nous vous présentons les informations à connaître.

[F35134]
[Congé du salarié pour le mariage de son [...]

✅ la fiche du mariage (F34154) est trouvée


- Le serveur est un **sous-processus** : le client le **démarre**, lui parle sur stdio, puis l'arrête.
- `tools/list` **découvre** les outils : le client n'a **rien codé** de notre RAG.
- Un outil renvoie du **texte** : ici les passages, chacun précédé de son identifiant de fiche.

In [3]:
try:
    async with stdio_client(server, errlog=devnull) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool("ask_fiches", {"question": "Combien de jours de congé pour mon mariage ?"})
            text = result.content[0].text
            print(text)
            print("\n✅ réponse sourcée avec la fiche F34154" if "F34154" in text else "\n❌ source attendue absente")
except Exception as error:
    print("⚠️ serveur, gateway ou Qdrant inaccessible :", type(error).__name__)
    print("Sortie enregistrée : « La durée de votre congé pour mariage est de 4 jours. » | Sources : F34154")

La durée de votre congé pour mariage est de 4 jours. Des dispositions conventionnelles peuvent prévoir une durée plus élevée.
Sources : F34154

✅ réponse sourcée avec la fiche F34154


- `ask_fiches` fait **tout le RAG** (recherche, génération, sources) derrière **un seul outil**.
- Deux niveaux d'outil : **`search_fiches`** (le client raisonne) ou **`ask_fiches`** (le serveur répond).

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 4. Un agent LangChain qui utilise le serveur </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- **`langchain-mcp-adapters`** convertit les outils MCP en **outils LangChain** : même `create_agent` qu'au chapitre 8.
- Aucun outil réécrit : l'agent **découvre** `search_fiches` et `ask_fiches`.

```
agent LangChain ──▶ outil LangChain (adaptateur) ──▶ tools/call ──▶ serveur MCP ──▶ Qdrant
```

In [4]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_openai import ChatOpenAI

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
mcp_client = MultiServerMCPClient({"fiches": {"transport": "stdio", "command": "uv",
                                              "args": ["run", "python", "-m", "src.mcp_server"], "cwd": str(Path.cwd())}})

try:
    async with mcp_client.session("fiches") as session:
        tools = await load_mcp_tools(session)
        print("Outils vus par l'agent :", [tool.name for tool in tools])
        agent = create_agent(llm, tools, system_prompt="Vous répondez aux questions de salariés sur leurs congés. "
                             "Cherchez dans les fiches avant de répondre, et citez l'identifiant des fiches.")
        state = await agent.ainvoke({"messages": [("user", "Combien de jours de congé pour enfant malade ?")]})

    calls = [call["name"] for message in state["messages"] for call in getattr(message, "tool_calls", None) or []]
    print("Appels d'outils :", calls)
    print("\n" + state["messages"][-1].content)
    print("\n✅ l'agent a appelé un outil MCP" if calls else "\n❌ aucun appel d'outil")
    print("(en direct, la formulation de la réponse varie)")
except Exception as error:
    print("⚠️ serveur, gateway ou Qdrant inaccessible :", type(error).__name__)
    print("Sortie enregistrée : outils ['search_fiches', 'ask_fiches'] | appel ['search_fiches'] | réponse : 3 jours par an "
          "(5 jours si l'enfant a moins d'un an ou si le salarié a au moins 3 enfants à charge), fiche F151.")

Outils vus par l'agent : ['search_fiches', 'ask_fiches']


Appels d'outils : ['search_fiches']

La durée du congé pour enfant malade dépend du nombre d'enfants à charge vivant au foyer du salarié :

1. **Pour un enfant de moins d'un an** : 5 jours par an.
2. **Pour au moins 3 enfants à charge de moins de 16 ans** : 5 jours par an.
3. **Dans les autres cas** : 3 jours par an.

Il est important de noter que la convention collective applicable peut prévoir une durée de congé plus longue. 

Ces informations proviennent de la fiche [F151](https://www.service-public.gouv.fr/particuliers/vosdroits/F151).

✅ l'agent a appelé un outil MCP
(en direct, la formulation de la réponse varie)


- L'agent a **choisi** l'outil et **cité** les fiches, sans une ligne d'intégration.
- Pourquoi `langchain-mcp-adapters` : c'est le **pont officiel** MCP → LangChain, et il garde **notre agent** (chapitre 8) tel quel.
- Un autre client (Claude Desktop) utilise **le même serveur**, sans changer une ligne.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > 5. Le brancher dans Claude Desktop </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

- On **déclare** le serveur dans un fichier de configuration : le client le **lance lui-même**.
- ⚠️ **Montré, pas exécuté** ici.

In [5]:
import json

config = {"mcpServers": {"fiches-conges": {
    "command": "uv",
    "args": ["--directory", "<chemin du dépôt>", "run", "python", "-m", "src.mcp_server"]}}}
print("~/.config/Claude/claude_desktop_config.json  (macOS : ~/Library/Application Support/Claude/)\n")
print(json.dumps(config, indent=2, ensure_ascii=False))
print("\nClaude Code :  claude mcp add fiches-conges -- uv --directory <chemin du dépôt> run python -m src.mcp_server")

~/.config/Claude/claude_desktop_config.json  (macOS : ~/Library/Application Support/Claude/)

{
  "mcpServers": {
    "fiches-conges": {
      "command": "uv",
      "args": [
        "--directory",
        "<chemin du dépôt>",
        "run",
        "python",
        "-m",
        "src.mcp_server"
      ]
    }
  }
}

Claude Code :  claude mcp add fiches-conges -- uv --directory <chemin du dépôt> run python -m src.mcp_server


- `--directory` : le serveur démarre **à la racine du projet** (`<chemin du dépôt>` = le dossier cloné), il y retrouve `data/qdrant/` et `.env`.
- **La clé n'est jamais dans la configuration** : elle reste dans `.env`.
- Après un redémarrage de Claude Desktop, les outils apparaissent : « Combien de jours pour mon mariage ? » **appelle `search_fiches`**.
- ⚠️ Qdrant local n'accepte **qu'un processus à la fois** : un client à la fois sur ce serveur.

<hr style="border-width:2px;border-color:#75DFC1">
<center><h2 style = "text-align:center" > Récapitulatif </h2></center>
<hr style="border-width:2px;border-color:#75DFC1">

| Ce qu'on a vu | À retenir |
|---|---|
| **Le protocole** | client ↔ serveur, `tools/list` puis `tools/call`, en JSON-RPC |
| **Le serveur** (`src/mcp_server.py`) | des fonctions Python + `@mcp.tool()`, lancé par `make mcp` |
| **Un client Python** | découvre les outils tout seul, appelle la recherche |
| **Un agent LangChain** | `langchain-mcp-adapters` : les outils MCP deviennent des outils LangChain |
| **Claude Desktop** | un bloc JSON `mcpServers` ; la clé reste dans `.env` |

- **Écrire l'outil une fois, le brancher partout.**
- **Démo seule** : pas de lab ni de chapitre projet pour MCP.
- **Prochain pas : la mise en production, Part 13.**